# Python Crash Course for Remote Sensing

This short notebook covers some basic Python skills for remote sensing. It uses dummy data, so you can run it immediately without downloading any satellite imagery.

## 1. Variables, Strings & File Paths
Store data and handle cross-platform file paths safely.

In [ ]:
from pathlib import Path

band_name = "B04"
resolution = 10
# Formatted strings are used to print out variables.
print(f"Loading {band_name} at {resolution}m resolution.")

# pathlib safely handles paths on Mac, Windows, and Linux using the '/' operator
data_dir = Path("data")
file_path = data_dir / "IMG_DATA" / f"{band_name}.jp2"
print(f"Expected file path: {file_path}")

## 2. Lists & Functions
Group items together and create reusable blocks of code.

In [ ]:
# A list of string filenames
bands_10m = ["B02", "B03", "B04", "B08"]

# A function to convert raw Sentinel-2 digital numbers to reflectance values.
def convert_dn_to_reflectance(dn_value):
    """Converts raw digital number to reflectance."""
    return (dn_value - 1000) / 10000.0

# Test the function
print(f"Reflectance for DN 2500: {convert_dn_to_reflectance(2500)}")

## 3. NumPy Arrays & Spatial Indexing
Satellite images are processed as multi-dimensional grids of numbers (arrays). Python uses 0-based indexing (the first item is index 0).

In [ ]:
import numpy as np

# Seed the random number generator for deterministic behaviour
np.random.seed(123)

# Create a dummy 3D array (100 rows, 100 cols, 4 bands) representing a satellite image
image_data = np.random.randint(1000, 20000, size=(100, 100, 4))

print(f"Array shape: {image_data.shape} (rows, cols, bands)")
print(f"Data type: {image_data.dtype}\n")

# Extract Band 0 (the very first band in the 3rd dimension)
band_0 = image_data[:, :, 0]
print(f"Band 0 shape: {band_0.shape}")

# Spatial slicing: crop a 10x10 pixel area from the top-left
crop = band_0[0:10, 0:10]
print(f"Crop shape: {crop.shape}")

## 4. Array Math & Logic
NumPy allows you to perform fast mathematical operations on entire images instantly.

In [ ]:
# Convert data to float32 and apply the Sentinel-2 correction formula
reflectance = (image_data.astype(np.float32) - 1000) / 10000.0

# Clip values so they stay between 0 and 1 for proper visualisation
reflectance_clipped = np.clip(reflectance, 0, 1)

print(f"Max value before clip: {reflectance.max():.3f}")
print(f"Max value after clip: {reflectance_clipped.max():.3f}\n")


# Safe division using np.where (prevents crashing from divide-by-zero errors)
nir = np.array([0.8, 0.0, 0.5])
red = np.array([0.2, 0.0, 0.1])
denominator = nir + red

ndvi = np.where(
    denominator != 0,
    (nir - red) / denominator,
    0.0  # Output 0.0 if the denominator is 0
)
print(f"Calculated NDVI (safe division): {ndvi}")

ndvi_inf = (nir - red) / denominator
# Without safe division, inf or nan will be returned
print(f"Calculated NDVI (non-safe division): {ndvi_inf}")

## 5. Quick Image Preview
Use matplotlib to plot arrays visually, mimicking the side-by-side previews in the main tutorial.

In [ ]:
import matplotlib.pyplot as plt

# Create a figure with 2 subplots side-by-side
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Plot a single band (Grayscale)
axes[0].set_title("Single Band (Band 0)")
axes[0].imshow(reflectance_clipped[:, :, 0], cmap='gray')
axes[0].set_xlabel("pixels")

# Plot a 3-band composite (RGB) using bands 0, 1, 2
axes[1].set_title("RGB Composite")
axes[1].imshow(reflectance_clipped[:, :, 0:3])
axes[1].set_xlabel("pixels")

plt.tight_layout()
plt.show()